# ASR Benchmark: Zipformer-30M (baseline) vs PhoWhisper
So sánh WER + RTF trên 604 câu hỏi dinh dưỡng tiếng Việt.

| Model | Framework | Device | Ghi chú |
|-------|-----------|--------|----------|
| **Zipformer-30M RNNT** (baseline) | sherpa-onnx | CPU | Model đang deploy |
| **PhoWhisper-small** | faster-whisper | GPU | 244M params |
| **PhoWhisper-medium** | faster-whisper | GPU | 769M params |
| **PhoWhisper-large** | faster-whisper | GPU | 1542M params |

**Upload lên Drive trước khi chạy:**
```
callbot_asr/
├── rnnt/
│   ├── encoder-epoch-20-avg-10.onnx   (~88MB)
│   ├── decoder-epoch-20-avg-10.onnx   (~5MB)
│   ├── joiner-epoch-20-avg-10.onnx    (~4MB)
│   └── config.json                    (tokens)
├── wav_16k/
│   └── {eval_1..5}/*.wav              (604 files)
└── eval_splits/
    └── eval_split_{1..5}.jsonl
```

In [ ]:
%%capture
!pip install sherpa-onnx faster-whisper jiwer soundfile

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────
DRIVE_BASE  = "/content/drive/MyDrive/callbot_asr"
RNNT_DIR    = f"{DRIVE_BASE}/rnnt"
WAV_ROOT    = f"{DRIVE_BASE}/wav_16k"
EVAL_DIR    = f"{DRIVE_BASE}/eval_splits"
RESULTS_DIR = "/content/asr_results"

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

# Uncomment nếu upload file zip
# !unzip -q {DRIVE_BASE}/wav_16k.zip -d {DRIVE_BASE}/
print("Paths OK")

In [ ]:
# ── Load reference transcripts ─────────────────────────────────────
import json
from pathlib import Path

def load_refs(eval_dir):
    refs = {}
    for i in range(1, 6):
        p = Path(eval_dir) / f"eval_split_{i}.jsonl"
        if not p.exists():
            continue
        with open(p) as f:
            for line in f:
                item = json.loads(line)
                refs[item["id"]] = {"question": item["question"], "source": item.get("source", "")}
    return refs

refs = load_refs(EVAL_DIR)
print(f"Loaded {len(refs)} references")

def build_items(wav_root, refs):
    items = []
    for wav in sorted(Path(wav_root).rglob("*.wav")):
        if wav.stem in refs:
            items.append({
                "path": wav,
                "id": wav.stem,
                "ref": refs[wav.stem]["question"],
                "source": refs[wav.stem]["source"],
            })
    print(f"Found {len(items)} wav files with references")
    return items

items = build_items(WAV_ROOT, refs)

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────
import re
import wave
import time
import numpy as np
from jiwer import wer as jiwer_wer

def get_duration(path):
    with wave.open(str(path), "rb") as f:
        return f.getnframes() / f.getframerate()

def normalize(text):
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def run_benchmark(transcribe_fn, items, model_name):
    print(f"\nBenchmarking {model_name} ({len(items)} files)...")
    results = []
    for i, item in enumerate(items):
        dur = get_duration(item["path"])
        t0 = time.perf_counter()
        hyp = transcribe_fn(item["path"])
        elapsed = time.perf_counter() - t0
        results.append({
            "id": item["id"],
            "source": item["source"],
            "ref": item["ref"],
            "hyp": hyp,
            "duration_s": dur,
            "inference_ms": elapsed * 1000,
            "rtf": elapsed / dur,
        })
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(items)} done")
    return results

def compute_metrics(results, model_name):
    refs_ = [normalize(r["ref"]) for r in results]
    hyps_ = [normalize(r["hyp"]) for r in results]
    overall_wer = jiwer_wer(refs_, hyps_)

    rtf = [r["rtf"] for r in results]
    inf = [r["inference_ms"] for r in results]

    wer_by_src = {}
    for src in sorted({r["source"] for r in results}):
        sub = [r for r in results if r["source"] == src]
        wer_by_src[src] = jiwer_wer([normalize(r["ref"]) for r in sub],
                                     [normalize(r["hyp"]) for r in sub])

    buckets = {"0-5s": [], "5-10s": [], ">10s": []}
    for r in results:
        if r["duration_s"] <= 5:
            buckets["0-5s"].append(r)
        elif r["duration_s"] <= 10:
            buckets["5-10s"].append(r)
        else:
            buckets[">10s"].append(r)

    wer_by_dur = {}
    for k, v in buckets.items():
        if v:
            wer_by_dur[k] = jiwer_wer([normalize(r["ref"]) for r in v],
                                       [normalize(r["hyp"]) for r in v])

    return {
        "model": model_name,
        "wer": overall_wer,
        "rtf_mean": np.mean(rtf),
        "rtf_p50": np.percentile(rtf, 50),
        "rtf_p90": np.percentile(rtf, 90),
        "latency_mean_ms": np.mean(inf),
        "latency_p90_ms": np.percentile(inf, 90),
        "wer_by_source": wer_by_src,
        "wer_by_duration": wer_by_dur,
        "rtf_values": rtf,
        "inf_values": inf,
        "dur_values": [r["duration_s"] for r in results],
    }

all_metrics = {}

## Model 1: Zipformer-30M RNNT (baseline — CPU)

In [ ]:
import sherpa_onnx
import soundfile as sf

rnnt = sherpa_onnx.OfflineRecognizer.from_transducer(
    encoder=f"{RNNT_DIR}/encoder-epoch-20-avg-10.onnx",
    decoder=f"{RNNT_DIR}/decoder-epoch-20-avg-10.onnx",
    joiner=f"{RNNT_DIR}/joiner-epoch-20-avg-10.onnx",
    tokens=f"{RNNT_DIR}/config.json",
    num_threads=4,
    provider="cpu",
)

def transcribe_rnnt(path):
    audio, sr = sf.read(str(path), dtype="float32")
    stream = rnnt.create_stream()
    stream.accept_waveform(sr, audio)
    rnnt.decode_stream(stream)
    return stream.result.text.strip()

results_rnnt = run_benchmark(transcribe_rnnt, items, "Zipformer-30M")
all_metrics["Zipformer-30M"] = compute_metrics(results_rnnt, "Zipformer-30M")

del rnnt
import gc; gc.collect()
print(f"Zipformer-30M: WER={all_metrics['Zipformer-30M']['wer']*100:.1f}%  RTF={all_metrics['Zipformer-30M']['rtf_mean']:.3f}")

## Model 2: PhoWhisper-small (GPU)

In [ ]:
from faster_whisper import WhisperModel

pw_small = WhisperModel("vinai/PhoWhisper-small", device="cuda", compute_type="float16")

def transcribe_pw(model, path):
    segs, _ = model.transcribe(str(path), language="vi", beam_size=5)
    return " ".join(s.text for s in segs).strip()

results_small = run_benchmark(lambda p: transcribe_pw(pw_small, p), items, "PhoWhisper-small")
all_metrics["PhoWhisper-small"] = compute_metrics(results_small, "PhoWhisper-small")

del pw_small
import torch; torch.cuda.empty_cache()
gc.collect()
print(f"PhoWhisper-small: WER={all_metrics['PhoWhisper-small']['wer']*100:.1f}%  RTF={all_metrics['PhoWhisper-small']['rtf_mean']:.3f}")

## Model 3: PhoWhisper-medium (GPU)

In [ ]:
pw_medium = WhisperModel("vinai/PhoWhisper-medium", device="cuda", compute_type="float16")

results_medium = run_benchmark(lambda p: transcribe_pw(pw_medium, p), items, "PhoWhisper-medium")
all_metrics["PhoWhisper-medium"] = compute_metrics(results_medium, "PhoWhisper-medium")

del pw_medium
torch.cuda.empty_cache()
gc.collect()
print(f"PhoWhisper-medium: WER={all_metrics['PhoWhisper-medium']['wer']*100:.1f}%  RTF={all_metrics['PhoWhisper-medium']['rtf_mean']:.3f}")

## Model 4: PhoWhisper-large (GPU)

In [ ]:
pw_large = WhisperModel("vinai/PhoWhisper-large", device="cuda", compute_type="float16")

results_large = run_benchmark(lambda p: transcribe_pw(pw_large, p), items, "PhoWhisper-large")
all_metrics["PhoWhisper-large"] = compute_metrics(results_large, "PhoWhisper-large")

del pw_large
torch.cuda.empty_cache()
gc.collect()
print(f"PhoWhisper-large: WER={all_metrics['PhoWhisper-large']['wer']*100:.1f}%  RTF={all_metrics['PhoWhisper-large']['rtf_mean']:.3f}")

## Kết quả

In [ ]:
# ── Summary table ──────────────────────────────────────────────────
print(f"{'Model':<20} {'WER':>6} {'RTF_mean':>9} {'RTF_p90':>8} {'Lat_mean':>9} {'Lat_p90':>8}")
print("-" * 65)
for name, m in all_metrics.items():
    print(f"{name:<20} {m['wer']*100:>5.1f}% {m['rtf_mean']:>9.3f} {m['rtf_p90']:>8.3f} "
          f"{m['latency_mean_ms']:>8.0f}ms {m['latency_p90_ms']:>7.0f}ms")

print("\n── WER theo source ──────────────────────────────────────────")
sources = sorted({s for m in all_metrics.values() for s in m["wer_by_source"]})
print(f"{'Model':<20}", end="")
for s in sources:
    print(f" {s:>15}", end="")
print()
for name, m in all_metrics.items():
    print(f"{name:<20}", end="")
    for s in sources:
        v = m["wer_by_source"].get(s)
        print(f" {v*100:>14.1f}%" if v is not None else f" {'N/A':>15}", end="")
    print()

print("\n── WER theo độ dài audio ────────────────────────────────────")
dur_keys = ["0-5s", "5-10s", ">10s"]
print(f"{'Model':<20}", end="")
for k in dur_keys:
    print(f" {k:>8}", end="")
print()
for name, m in all_metrics.items():
    print(f"{name:<20}", end="")
    for k in dur_keys:
        v = m["wer_by_duration"].get(k)
        print(f" {v*100:>7.1f}%" if v is not None else f" {'N/A':>8}", end="")
    print()

In [ ]:
# ── Save CSV ───────────────────────────────────────────────────────
import csv
from datetime import datetime

tag = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = f"{RESULTS_DIR}/summary_{tag}.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["model", "wer", "rtf_mean", "rtf_p90", "latency_mean_ms", "latency_p90_ms"])
    w.writeheader()
    for name, m in all_metrics.items():
        w.writerow({k: m[k] for k in w.fieldnames})
print(f"Saved: {csv_path}")

In [ ]:
# ── Charts ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

models = list(all_metrics.keys())
colors = ["#4e79a7", "#f28e2b", "#59a14f", "#e15759"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("ASR Benchmark: Zipformer-30M vs PhoWhisper (tiếng Việt, 604 files)",
             fontsize=13, fontweight="bold")

# 1. WER bar
ax = axes[0, 0]
wers = [all_metrics[m]["wer"] * 100 for m in models]
bars = ax.bar(models, wers, color=colors[:len(models)], edgecolor="white", linewidth=0.5)
ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=10)
ax.set_title("Word Error Rate (WER) — thấp hơn tốt hơn")
ax.set_ylabel("WER (%)")
ax.set_ylim(0, max(wers) * 1.2)
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.3)

# 2. RTF CDF
ax = axes[0, 1]
for i, (name, m) in enumerate(all_metrics.items()):
    sorted_rtf = sorted(m["rtf_values"])
    cdf = np.arange(1, len(sorted_rtf) + 1) / len(sorted_rtf)
    ax.plot(sorted_rtf, cdf, label=name, color=colors[i], linewidth=2)
ax.axvline(1.0, color="red", linestyle="--", linewidth=1, alpha=0.6, label="RTF=1 (real-time)")
ax.set_title("CDF của RTF — đường càng trái càng nhanh")
ax.set_xlabel("RTF")
ax.set_ylabel("Tỷ lệ tích lũy")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# 3. Audio duration vs inference time scatter
ax = axes[1, 0]
for i, (name, m) in enumerate(all_metrics.items()):
    ax.scatter(m["dur_values"], m["inf_values"], alpha=0.4, s=15,
               color=colors[i], label=name)
ax.set_title("Độ dài audio vs Thời gian xử lý")
ax.set_xlabel("Audio duration (s)")
ax.set_ylabel("Inference time (ms)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# 4. WER by duration bucket
ax = axes[1, 1]
dur_keys = ["0-5s", "5-10s", ">10s"]
x = np.arange(len(dur_keys))
width = 0.2
for i, name in enumerate(models):
    vals = [all_metrics[name]["wer_by_duration"].get(k, 0) * 100 for k in dur_keys]
    ax.bar(x + i * width, vals, width, label=name, color=colors[i], edgecolor="white")
ax.set_title("WER theo độ dài audio")
ax.set_ylabel("WER (%)")
ax.set_xticks(x + width * (len(models) - 1) / 2)
ax.set_xticklabels(dur_keys)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
chart_path = f"{RESULTS_DIR}/charts_{tag}.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {chart_path}")

In [ ]:
# ── Copy về Drive ───────────────────────────────────────────────────
!mkdir -p {DRIVE_BASE}/results
!cp {RESULTS_DIR}/* {DRIVE_BASE}/results/
print(f"Đã copy kết quả về: {DRIVE_BASE}/results/")